# OSNet (AIN 미적용) 크로스 도메인 Ablation 실험

**목적**: 실험 2 (`02_osnet_ain_crossdomain.ipynb`) 에서 OSNet-AIN 이 Market→Duke 에서 Rank-1 53.5% 를 기록한 것이 **AIN(Instance Normalization) 덕분인지**, 아니면 OSNet 의 omni-scale 구조 자체가 원래 일반화에 강한 것인지 구분하기 위한 ablation 실험임.

**비교 설계**

| 모델 | Market→Market (실험1) | Market→Duke (실험2 / 본 실험) |
|------|----------------------|------------------------------|
| OSNet (AIN 없음) | 94.4% Rank-1 | **본 실험에서 측정** |
| OSNet-AIN | — | 53.5% Rank-1 |

두 모델 모두 Market-1501 가중치를 그대로 사용하고, 평가만 Duke 로 바꿔 진행함. 모델만 다르고 다른 조건은 동일하므로 차이는 AIN 의 기여로 해석할 수 있음.

In [1]:
# OSNet (AIN 미적용) 크로스 도메인 Ablation
# 논문: Learning Generalisable Omni-Scale Representations for Person Re-Identification
# IEEE TPAMI 2021

import torchreid
import torch

print(f"torchreid 버전: {torchreid.__version__}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")

torchreid 버전: 1.4.0
PyTorch 버전: 2.5.1+cu121
GPU 사용 가능: True


In [2]:
# 데이터셋 로딩
# 실험 2와 동일하게 Market-1501 학습 / Duke 평가 (크로스 도메인)
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

datamanager = torchreid.data.ImageDataManager(
    root='/home/ubuntu/datasets',
    sources='market1501',         # 학습 데이터셋
    targets='dukemtmcreid',        # 테스트 데이터셋 (크로스 도메인!)
    height=256,
    width=128,
    batch_size_train=32,
    batch_size_test=32,
    transforms=['random_flip', 'random_crop']
)

Building train transforms ...
+ resize to 256x128
+ random flip
+ random crop (enlarge to 288x144 and crop 256x128)
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded DukeMTMCreID
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   702 |    16522 |         8
  query    |   702 |     2228 |         8
  g

In [3]:
# OSNet 모델 로딩 (AIN 미적용 / Market-1501 에서 학습된 가중치)
# 실험 2와의 유일한 차이: 모델 이름이 'osnet_x1_0' (실험 2는 'osnet_ain_x1_0')
model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=751,
    pretrained=False
)

torchreid.utils.load_pretrained_weights(
    model,
    '/home/ubuntu/model/osnet_x1_0_market_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip.pth'
)

model = model.cuda()
print("OSNet (AIN 미적용) 모델 로딩 완료!")

Successfully loaded pretrained weights from "/home/ubuntu/model/osnet_x1_0_market_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip.pth"
OSNet (AIN 미적용) 모델 로딩 완료!


/home/ubuntu/deep-person-reid/torchreid/utils/torchtools.py:85: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fpath, map_location=map_location)


In [4]:
# 크로스 도메인 mAP 측정 (Market-1501 학습 → Duke 평가)
engine = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model,
    optimizer=torchreid.optim.build_optimizer(model, optim='adam', lr=0.0003)
)

engine.run(
    test_only=True,
    visrank=False
)

##### Evaluating dukemtmcreid (target) #####
Extracting features from query set ...
Done, obtained 2228-by-512 matrix
Extracting features from gallery set ...
Done, obtained 14588-by-512 matrix
Speed: 0.0084 sec/batch
Computing distance matrix with metric=euclidean ...
Computing CMC and mAP ...
** Results **
mAP: 21.0%
CMC curve
Rank-1  : 37.5%
Rank-5  : 53.4%
Rank-10 : 60.3%
Rank-20 : 65.8%


In [ ]:
import matplotlib.pyplot as plt

# Ablation 비교: OSNet (AIN 없음) vs OSNet-AIN, 둘 다 Market→Duke 크로스 도메인
models_compared = ['OSNet\n(no AIN)', 'OSNet-AIN']
rank1 = [37.5, 53.5]
mAP_values = [21.0, 31.1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(models_compared, rank1, color=['#A53D18', '#0F6E56'])
ax1.set_title('Rank-1 (Market->Duke)')
ax1.set_ylabel('Accuracy (%)')
for i, v in enumerate(rank1):
    if v is not None:
        ax1.text(i, v+0.3, f'{v}%', ha='center')

ax2.bar(models_compared, mAP_values, color=['#A53D18', '#0F6E56'])
ax2.set_title('mAP (Market->Duke)')
ax2.set_ylabel('mAP (%)')
for i, v in enumerate(mAP_values):
    if v is not None:
        ax2.text(i, v+0.3, f'{v}%', ha='center')

plt.suptitle('Ablation: AIN의 Cross-Domain 일반화 기여도')
plt.tight_layout()
plt.savefig('osnet_ablation_results.png')
plt.show()

## 결과 해석 가이드

위 셀의 OSNet (AIN 미적용) Rank-1 결과를 OSNet-AIN 의 53.5% 와 비교하여 다음과 같이 해석함.

- **OSNet < OSNet-AIN 으로 큰 차이**: AIN 이 cross-domain 일반화의 주된 동력임이 입증됨. 논문 주장과 일치함.
- **OSNet ≈ OSNet-AIN**: 일반화 성능의 대부분은 omni-scale 구조 자체에서 나온 것이며, AIN 의 추가 효과는 제한적임. 논문 주장에 의문 부호가 붙음.
- **OSNet > OSNet-AIN**: 매우 이례적. AIN 이 오히려 source 도메인 정보를 과도하게 제거했을 가능성이 있음.

어느 결과가 나오든, 단순 성능 보고가 아닌 **원인-결과 분석**을 발표에 담을 수 있음.